In [1]:
import pandas as pd
from transformers import BartTokenizer, BartForConditionalGeneration, Trainer, TrainingArguments
from datasets import Dataset

In [2]:
train_data= pd.read_csv("../Data/samsum-train.csv")
val_data = pd.read_csv("../Data/samsum-validation.csv")

### Data- Preprocessing

In [3]:
import re

def clean_data(text):
    text = re.sub(r"\r\n"," ",text)
    text = re.sub(r"\s+"," ",text)
    text = re.sub(r"<.*?>"," ",text)
    text = text.strip().lower()
    return text 

In [4]:
train_data["dialogue"] = train_data["dialogue"].fillna("").astype(str).apply(clean_data)
train_data["summary"] = train_data["summary"].fillna("").astype(str).apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].fillna("").astype(str).apply(clean_data)
val_data["summary"] = val_data["summary"].fillna("").astype(str).apply(clean_data)

In [5]:
train_data["dialogue"][0]

"amanda: i baked cookies. do you want some? jerry: sure! amanda: i'll bring you tomorrow :-)"

### Tokenize

In [6]:
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")

In [7]:
# raw data =>tokenize 

def tokenize(data):
    inputs = tokenizer(
        data["dialogue"],
        padding="max_length",
        max_length=128,
        truncation=True
    )
    
    targets = tokenizer(
        data["summary"],
        padding="max_length",
        max_length=32,
        truncation=True
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

In [8]:


train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)

train_dataset = train_dataset.map(tokenize, batched=True) 
val_dataset = val_dataset.map(tokenize, batched=True)


Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

### Working with the Model

In [9]:
import torch 

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(device)

cuda


In [10]:
#NLP =>Generation task


model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")
model.to(device)

BartForConditionalGeneration(
  (model): BartModel(
    (shared): Embedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartSdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (final_l

In [13]:
training_args = TrainingArguments(
    output_dir="./results",

    # Core (keep stable)
    num_train_epochs=5,  
    weight_decay=0.01,
    learning_rate=3e-5,
    per_device_train_batch_size=4,   
    per_device_eval_batch_size=4,
    gradient_accumulation_steps = 2,
   
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=500,
    warmup_steps=200,
    fp16=True,                     
    dataloader_num_workers=0,    
    save_total_limit=2,
    report_to="none",

    load_best_model_at_end=True,   
    metric_for_best_model="eval_loss",  
    greater_is_better=False
)

In [14]:
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

d:\Vs code\AIML\.venv\Lib\site-packages\accelerate\accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [15]:
#Train the module
trainer.train()

  0%|          | 0/9205 [00:00<?, ?it/s]

{'loss': 1.5901, 'grad_norm': 6.742351055145264, 'learning_rate': 2.9010549694614105e-05, 'epoch': 0.27}
{'loss': 1.1654, 'grad_norm': 5.806622505187988, 'learning_rate': 2.7344808439755692e-05, 'epoch': 0.54}
{'loss': 1.1317, 'grad_norm': 5.881405353546143, 'learning_rate': 2.567906718489728e-05, 'epoch': 0.81}


  0%|          | 0/205 [00:00<?, ?it/s]

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}


{'eval_loss': 1.1372641324996948, 'eval_runtime': 13.391, 'eval_samples_per_second': 61.086, 'eval_steps_per_second': 15.309, 'epoch': 1.0}
{'loss': 1.0048, 'grad_norm': 5.8203630447387695, 'learning_rate': 2.4013325930038867e-05, 'epoch': 1.09}
{'loss': 0.8292, 'grad_norm': 5.463254451751709, 'learning_rate': 2.2350916157690172e-05, 'epoch': 1.36}
{'loss': 0.8351, 'grad_norm': 4.556014537811279, 'learning_rate': 2.0685174902831763e-05, 'epoch': 1.63}
{'loss': 0.8329, 'grad_norm': 4.470837593078613, 'learning_rate': 1.9019433647973347e-05, 'epoch': 1.9}


...

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}


{'eval_loss': 1.3485387563705444, 'eval_runtime': 70.8166, 'eval_samples_per_second': 11.551, 'eval_steps_per_second': 2.895, 'epoch': 4.0}


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


{'train_runtime': 70591.1048, 'train_samples_per_second': 1.043, 'train_steps_per_second': 0.13, 'train_loss': 0.7800481352442281, 'epoch': 4.0}


TrainOutput(global_step=7366, training_loss=0.7800481352442281, metrics={'train_runtime': 70591.1048, 'train_samples_per_second': 1.043, 'train_steps_per_second': 0.13, 'total_flos': 1.5962892499156992e+16, 'train_loss': 0.7800481352442281, 'epoch': 4.0})

In [20]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}


('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\special_tokens_map.json',
 './saved_summary_model\\vocab.json',
 './saved_summary_model\\merges.txt',
 './saved_summary_model\\added_tokens.json')

In [21]:
model = BartForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = BartTokenizer.from_pretrained("./saved_summary_model")

### Test the core logic for summarization

In [22]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) #clean

    #tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length = 512,
        truncation = True,
        return_tensors = "pt"
    ).to(device)

    #generate the summary =>token ids
    model.to(device)
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 128,
        num_beams= 4,
        early_stopping = True
    )


    #Token ids convert to summary => decoding
    summary = tokenizer.decode(targets[0],skip_special_tokens=True)
    return summary

In [23]:
test_dialogue = """Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)"""

summary = summarize_dialogue(test_dialogue)
print("Summary:" ,summary)

Summary: eric, rob and eric are watching some of his stand-up comedy on youtube. they like the train part of it very much. eric and rob are going to watch the stand-ups on youtube with the ttyl. and the e.
